# 03a Build `apn_firmcode.dta` (slot-level)

This notebook builds `${stata_output}apn_firmcode.dta` from `${CN_patents}businfo.dta`.

Rules implemented:
- `;` separates co-applicant slots (empty slots are meaningful and preserved)
- `,` may separate multiple stock codes within one slot (keep only the first)
- extract stock suffix (e.g. `SH`, `SZ`, `HK`) into `stock_suffix`
- final output keeps only `[apn, SocialCreditCode, Symbol]`
- drop rows where `stock_suffix == "HK"` or `Symbol` is missing


In [ ]:
from pathlib import Path
import re
import pandas as pd

BUSINFO_PATH = Path("../../../dataset/CN_patents/1985-2025/businfo.dta")
OUTPUT_PATH = Path("../../../stata_output/CN_CN/apn_firmcode.dta")
CHUNKSIZE = 200_000

print("businfo:", BUSINFO_PATH.resolve())
print("output :", OUTPUT_PATH.resolve())
print("chunksize:", CHUNKSIZE)

In [ ]:
credit_pat = re.compile(r"[0-9A-Z]{18}")
# Capture a single stock code: 5 digits (HK) or 6 digits; optional suffix.
first_stock_pat = re.compile(r"(?P<code>\d{5,6})(?:\.(?P<suffix>[A-Z]{2}))?")

def s(x):
    return "" if pd.isna(x) else str(x)

def parse_businfo_slots_keep_first_stock(df):
    rows = []
    for apn, credit_raw, stock_raw in df[["申请号", "工商统一社会信用代码", "工商上市代码"]].itertuples(index=False, name=None):
        credit_slots = [x.strip() for x in s(credit_raw).split(";")]  # keep empty slots
        stock_slots = [x.strip() for x in s(stock_raw).split(";")]    # keep empty slots

        n = max(len(credit_slots), len(stock_slots))
        credit_slots += [""] * (n - len(credit_slots))
        stock_slots += [""] * (n - len(stock_slots))

        for slot_idx, (cslot, sslot) in enumerate(zip(credit_slots, stock_slots), start=1):
            c_norm = cslot.upper().replace(" ", "")
            s_norm = sslot.upper().replace(" ", "")

            credit_codes = credit_pat.findall(c_norm)
            social_credit_code = credit_codes[0] if credit_codes else None

            # A slot may contain multiple codes (e.g., "000725,200725"); keep the first match only.
            m = first_stock_pat.search(s_norm)
            if m:
                stock_code = m.group("code")          # numeric only (no suffix)
                stock_suffix = m.group("suffix")      # SZ / SH / HK / None
                is_listed_slot = 1
            else:
                stock_code = None
                stock_suffix = None
                is_listed_slot = 0

            rows.append({
                "apn": str(apn).strip(),
                "applicant_slot": slot_idx,
                "social_credit_code": social_credit_code,
                "stock_code": stock_code,             # numeric only (no suffix)
                "stock_suffix": stock_suffix,         # extracted suffix
                "stock_codes_raw": sslot,             # keep raw slot text for traceability
                "is_listed_slot": is_listed_slot,
                "flag_missing_credit": int(social_credit_code is None),
            })

    return pd.DataFrame(rows)

print("Parser functions ready")

In [ ]:
# Read + build in a single pass over businfo.dta (read_stata is called only once)
want_apn = {
    "200810239807",  # 4 applicants, only one listed in slot 3
    "200810094583",  # leading semicolon slot
    "200810231591",  # two listed applicants
    "201910849469",  # middle slot with comma multi-codes
    "202180000461",  # first slot comma multi-codes + trailing semicolon
    "200610125651",  # simple no suffix
    "200510129587",  # simple SH suffix
    "200710011705",  # simple HK suffix
}

raw_preview = None
example_parts = []
seen_apn = set()

out_parts = []
total_in_rows = 0
total_out_rows = 0

reader = pd.read_stata(
    BUSINFO_PATH,
    columns=["申请号", "工商统一社会信用代码", "工商上市代码"],
    chunksize=CHUNKSIZE,
)

for chunk_idx, chunk in enumerate(reader, start=1):
    total_in_rows += len(chunk)

    if raw_preview is None:
        raw_preview = chunk.head(10).copy()

    apn_series = chunk["申请号"].astype(str).str.strip()
    remaining = want_apn - seen_apn
    if remaining:
        mask = apn_series.isin(remaining)
        if mask.any():
            sub = chunk.loc[mask, ["申请号", "工商统一社会信用代码", "工商上市代码"]].copy()
            sub["申请号"] = apn_series[mask]
            example_parts.append(sub)
            seen_apn.update(sub["申请号"].unique().tolist())

    parsed = parse_businfo_slots_keep_first_stock(chunk)

    parsed = parsed[
        (parsed["stock_code"].notna())
        & (parsed["stock_code"] != "")
        & (parsed["stock_suffix"] != "HK")
    ]

    parsed = parsed.rename(columns={"social_credit_code": "SocialCreditCode", "stock_code": "Symbol"})
    parsed = parsed[["apn", "SocialCreditCode", "Symbol"]]

    total_out_rows += len(parsed)
    out_parts.append(parsed)

    if chunk_idx % 10 == 0:
        print(
            f"chunk={chunk_idx}, in_rows={total_in_rows:,}, out_rows={total_out_rows:,}, "
            f"examples_found={len(seen_apn)}/{len(want_apn)}"
        )

raw_examples = (
    pd.concat(example_parts, ignore_index=True)
    .drop_duplicates(subset=["申请号"])
    .sort_values("申请号")
    if example_parts
    else pd.DataFrame(columns=["申请号", "工商统一社会信用代码", "工商上市代码"])
)

parsed_examples = parse_businfo_slots_keep_first_stock(raw_examples) if not raw_examples.empty else pd.DataFrame()
if not parsed_examples.empty:
    parsed_examples = parsed_examples[
        (parsed_examples["stock_code"].notna())
        & (parsed_examples["stock_code"] != "")
        & (parsed_examples["stock_suffix"] != "HK")
    ]
    parsed_examples = parsed_examples.rename(
        columns={"social_credit_code": "SocialCreditCode", "stock_code": "Symbol"}
    )
    parsed_examples = parsed_examples[
        ["apn", "applicant_slot", "SocialCreditCode", "Symbol", "stock_suffix", "stock_codes_raw"]
    ]

result = (
    pd.concat(out_parts, ignore_index=True)
    if out_parts
    else pd.DataFrame(columns=["apn", "SocialCreditCode", "Symbol"])
)

result.to_stata(OUTPUT_PATH, write_index=False, version=118)

print(f"Saved {len(result):,} rows to {OUTPUT_PATH.resolve()}")
print("Saved columns:", list(result.columns))

# Preview raw contents (first rows)
raw_preview

In [ ]:
# Preview examples and results (no additional file reads)
print("Raw examples (captured during the single pass):")
display(raw_examples)

print("Parsed examples (filtered to final output):")
display(parsed_examples)

print("Result preview:")
display(result.head(20))

print("Result columns:", list(result.columns))
print("Result rows:", f"{len(result):,}")